# Week 40: integrated multilingual QA (Section 6, part 1)

Thin notebook over `src/qa_model.py` and `scripts/run_week40.py`, like the Week 36 one: all logic lives in the repo, so the numbers in `results/` are the ones the report reads.

**Runtime:** use a GPU (Runtime > Change runtime type > T4 GPU). Three seeds of XLM-R base take roughly 30 to 60 minutes on a T4 (a rough estimate, the run records the real time).

## 0. Setup

Pick one. `drive` is the least painful: results and the saved model survive a runtime restart.
* `drive`: the repo folder is in your Google Drive.
* `upload`: upload a zip of the repo (`diku-nlp-project.zip`).
* `github`: clone `lichtnatalie/diku-nlp-project` (the Week 40 files must be committed and pushed first; a private repo needs a token).

In [ ]:
SETUP = "drive"   # "drive" | "upload" | "github"
import os, subprocess

if SETUP == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/diku-nlp-project'      # <- adjust
elif SETUP == "upload":
    from google.colab import files
    up = files.upload()
    subprocess.run(['unzip', '-oq', next(iter(up)), '-d', '/content'], check=True)
    PROJECT_DIR = '/content/diku-nlp-project'
else:
    subprocess.run(['git', 'clone', 'https://github.com/lichtnatalie/diku-nlp-project', '/content/diku-nlp-project'], check=True)
    PROJECT_DIR = '/content/diku-nlp-project'

os.chdir(PROJECT_DIR)
print(os.getcwd())

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" pandas pyarrow scikit-learn tabulate pytest
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch the runtime to a GPU')

## 1. Tests first

The Week 40 tests run offline on a tiny random model: windows cover every context character, targets come from the stored offsets, the decoder only returns legal spans, a high no-answer score means abstention, the SQuAD-v2 metrics, and best-epoch selection keeps a real copy of the weights. All must pass before the real run.

In [ ]:
!python -m pytest -q tests/test_week40.py

## 2. The run

Downloads `FacebookAI/xlm-roberta-base`, measures sequence lengths, audits the training targets (and stops if any answer cannot be labelled), fine-tunes under three seeds with the epoch chosen on the Week 39 dev slice, tunes the no-answer threshold on validation, and writes `results/60_week40_*` to `results/66_week40_*` and `results/environment_week40.json`.

`--save-model` keeps the first seed's model, tokenizer and decision rule (threshold included). Section 8 needs the model and threshold **frozen** before the challenge set, so keep this folder.

In [ ]:
!python -m scripts.run_week40 --save-model models/week40_xlmr_seed20260901

## 3. Look at the results

In [ ]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
for name in ['60_week40_sequence_lengths', '61_week40_label_audit', '62_week40_epoch_selection']:
    print(name); display(pd.read_csv(f'results/{name}.csv'))
res = pd.read_csv('results/64_week40_qa_results.csv')
display(res[['threshold_kind', 'subset', 'n', 'exact_match', 'f1', 'has_ans_f1', 'no_ans_exact_match',
             'answerable_f1', 'unanswerable_f1', 'macro_f1', 'f1_sd', 'macro_f1_sd', 'threshold']].round(3))
display(pd.read_csv('results/65_week40_span_metrics_bio.csv').round(3))

## 4. Take the results home

If you used `drive`, the files are already in the repo folder on Drive: copy `results/6*_week40_*` and `results/environment_week40.json` into the repo on the laptop. Otherwise download the zip. The saved model stays on Colab or Drive (it is about 1 GB and is not committed).

In [ ]:
!zip -q -r week40_results.zip results/6*_week40_* results/environment_week40.json
from google.colab import files
files.download('week40_results.zip')